# 네이버 블로그 크롤러
구글 Colab에서 네이버 블로그 콘텐츠를 크롤링합니다.

## 1. 필수 라이브러리 설치

In [ ]:
# Chrome 및 ChromeDriver 설치 (구글 Colab용)
!apt-get update
!apt-get install -y chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin

# 필요한 Python 패키지 설치
!pip install selenium beautifulsoup4

## 2. 라이브러리 임포트

In [ ]:
import time
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

## 3. WebDriver 설정 함수

In [ ]:
def setup_driver():
    """
    Chrome WebDriver 설정 (구글 Colab용)
    """
    chrome_options = Options()
    chrome_options.add_argument('--headless')  # 백그라운드 실행
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/98.0.4758.102 Safari/537.36")

    driver = webdriver.Chrome(options=chrome_options)
    return driver

## 4. 블로그 콘텐츠 추출 함수

In [ ]:
def extract_blog_content(url, wait_time=5):
    """
    네이버 블로그 콘텐츠 추출

    Args:
        url (str): 네이버 블로그 URL
        wait_time (int): 페이지 로딩 대기 시간 (초)

    Returns:
        str: 추출된 블로그 텍스트 내용
    """
    driver = None
    try:
        # 드라이버 설정
        driver = setup_driver()

        # 페이지 열기
        print(f"블로그 URL 접속 중: {url}")
        driver.get(url)
        time.sleep(wait_time)

        # iframe 찾기 및 전환
        print("iframe으로 전환 중...")
        iframe = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "mainFrame"))
        )
        driver.switch_to.frame(iframe)

        # 페이지 소스 가져오기
        source = driver.page_source
        html = BeautifulSoup(source, "html.parser")

        # 콘텐츠 추출
        print("콘텐츠 추출 중...")
        content_container = html.select("div.se-main-container")

        if not content_container:
            print("경고: 콘텐츠를 찾을 수 없습니다.")
            return ""

        # 텍스트만 추출
        content = ''.join(str(content_container))

        # HTML 태그 제거
        pattern1 = '<[^>]*>'
        content = re.sub(pattern=pattern1, repl='', string=content)

        # 불필요한 패턴 제거
        pattern2 = """[\n\n\n\n\n// flash 오류를 우회하기 위한 함수 추가\nfunction _flash_removeCallback() {}"""
        content = content.replace(pattern2, '')

        # 텍스트 정제
        content = content.replace('\n', ' ')
        content = content.replace('\u200b', '')  # Zero-width space 제거
        content = content.replace('&ZeroWidthSpace;', '')

        # 연속된 공백 제거
        content = re.sub(r'\s+', ' ', content).strip()

        print("콘텐츠 추출 완료!")
        return content

    except Exception as e:
        print(f"에러 발생: {str(e)}")
        return ""

    finally:
        if driver:
            driver.quit()

## 5. 블로그 크롤링 실행

In [ ]:
# 크롤링할 블로그 URL
blog_url = "https://blog.naver.com/yminsong/224075787010"

# 콘텐츠 추출
print("=" * 80)
print("네이버 블로그 크롤링 시작")
print("=" * 80)

content = extract_blog_content(blog_url)

if content:
    print("\n" + "=" * 80)
    print("추출된 콘텐츠:")
    print("=" * 80)
    print(content)
    print("\n" + "=" * 80)
    print(f"총 텍스트 길이: {len(content)} 자")
    print("=" * 80)
else:
    print("콘텐츠를 추출하지 못했습니다.")

## 6. 여러 URL 크롤링하기 (선택사항)

In [ ]:
# 여러 개의 블로그 URL을 크롤링하고 싶다면 아래 코드를 사용하세요
naver_urls = [
    "https://blog.naver.com/yminsong/224075787010",
    # 추가 URL을 여기에 입력
]

contents = []

for url in naver_urls:
    print(f"\n처리 중: {url}")
    content = extract_blog_content(url)
    if content:
        contents.append({
            'url': url,
            'content': content,
            'length': len(content)
        })
    print("-" * 80)

# 결과 출력
print("\n" + "=" * 80)
print(f"총 {len(contents)}개의 블로그 글을 크롤링했습니다.")
print("=" * 80)

for idx, item in enumerate(contents, 1):
    print(f"\n[{idx}] {item['url']}")
    print(f"길이: {item['length']}자")
    print(f"내용 미리보기: {item['content'][:200]}...")